# 04 — Top 10 Most Profitable Products
Highest-profit products ranked with margin percentage, colored by category.


In [ ]:
import os, sys
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns

os.environ['JAVA_HOME']             = '/usr/local/java'
os.environ['SPARK_HOME']            = '/usr/local/spark'
os.environ['HADOOP_CONF_DIR']       = '/usr/local/hadoop/etc/hadoop'
os.environ['PYSPARK_PYTHON']        = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

try:
    spark.stop()
except:
    pass

# Local mode — no Hive/hive-metastore dependency
spark = (SparkSession.builder
    .appName("Superstore Analytics")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

# ── Load CSVs and register temp views ─────────────────────────────────────────
DATA = "/usr/local/hadoop/etc/hadoop/assessment-2"

def load(filename, renames):
    df = spark.read.option("header", "true").option("inferSchema", "true") \
             .csv(f"{DATA}/{filename}")
    for old, new in renames.items():
        df = df.withColumnRenamed(old, new)
    return df

customers  = load("customers.csv",  {"Customer ID": "customer_id",
                                      "Customer Name": "customer_name",
                                      "Segment": "segment"})
orders     = load("orders.csv",     {"Order ID": "order_id",
                                      "Order Date": "order_date",
                                      "Ship Date": "ship_date",
                                      "Ship Mode": "ship_mode",
                                      "Customer ID": "customer_id",
                                      "Postal Code": "postal_code"})
order_items = load("order_items.csv", {"Row ID": "row_id",
                                        "Order ID": "order_id",
                                        "Product ID": "product_id",
                                        "Sales": "sales",
                                        "Quantity": "quantity",
                                        "Discount": "discount",
                                        "Profit": "profit"})
products   = load("products.csv",   {"Product ID": "product_id",
                                      "Product Name": "product_name",
                                      "Category": "category",
                                      "Sub-Category": "sub_category"})
locations  = load("locations.csv",  {"Postal Code": "postal_code",
                                      "City": "city",
                                      "State": "state",
                                      "Country": "country",
                                      "Region": "region"})

customers.createOrReplaceTempView("customers")
orders.createOrReplaceTempView("orders")
order_items.createOrReplaceTempView("order_items")
products.createOrReplaceTempView("products")
locations.createOrReplaceTempView("locations")

print("Spark", spark.version, "ready — all tables loaded.")
spark.sql("SHOW TABLES").show()

# ── Global chart style ─────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
})
PALETTE = sns.color_palette("muted")

def fmt_usd(v):
    if abs(v) >= 1_000_000:
        return f"${v/1_000_000:.2f}M"
    if abs(v) >= 1_000:
        return f"${v/1_000:.1f}K"
    return f"${v:.0f}"


In [ ]:
# ── Top 10 Most Profitable Products ──────────────────────────────────────────
top_prod = spark.sql("""
    SELECT p.product_name,
           p.category,
           ROUND(SUM(oi.profit), 2)                    AS profit,
           ROUND(SUM(oi.profit)/SUM(oi.sales)*100, 2) AS margin_pct
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    GROUP BY p.product_name, p.category
    ORDER BY profit DESC
    LIMIT 10
""").toPandas()

cat_pal = {"Furniture": PALETTE[0], "Office Supplies": PALETTE[1],
           "Technology": PALETTE[2]}
colors  = [cat_pal.get(c, PALETTE[3]) for c in top_prod["category"][::-1]]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(top_prod["product_name"].str[:45][::-1],
               top_prod["profit"][::-1],
               color=colors, edgecolor="white")
for bar, margin in zip(bars, top_prod["margin_pct"][::-1]):
    ax.text(bar.get_width() + 30,
            bar.get_y() + bar.get_height()/2,
            f"{fmt_usd(bar.get_width())} ({margin}%)",
            va="center", fontsize=8)
handles = [mpatches.Patch(color=v, label=k) for k, v in cat_pal.items()]
ax.legend(handles=handles, frameon=False, fontsize=9)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: fmt_usd(v)))
ax.grid(axis="x", linestyle="--", alpha=0.4)
ax.set_xlabel("Profit (USD)")
ax.set_title("Top 10 Most Profitable Products")
plt.tight_layout()
plt.show()


In [ ]:
spark.stop()
print("Spark stopped.")
